# Bi-stable double well: analysis

Post-processing of the TPS simulations generated with `bistable_double_well_generate_data.ipynb`
(50 replicas per method and 250 replicas for the reference). This notebook produces the processed
data used in `figures.ipynb`:

| Output | Used in |
|---|---|
| `reference/twoway_shooting/bsdw/Uniform/Hist_ref_avg_for_plotting.txt` | Fig. 2B (main text) |
| `results/<algorithm>/bsdw/<p_sel>/ch_correlation_<run>.txt` | Fig. 2D (main text), Table S1 |
| `results/path_lengths/bsdw_P_L.txt` | Fig. S6 (right) |
| `results/efficiency_bsdw.txt` | Table S1 |

The path overlap (Fig. 2C) is computed in `bistable_double_well_path_overlap.ipynb`, the convergence
test (Fig. S2) with the scripts in `scripts/`.

In [ ]:
import numpy as np
import pickle
import os
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
from aaa_tps.systems.two_dimensional_double_wells import BistableDoubleWell
from aaa_tps.states.states import circular_state
from aaa_tps.utils.observables import get_weighted_autocorrelation_function, relative_effective_sample_size

In [ ]:
bsdw = BistableDoubleWell(A=15, B=0.25)

bsdw_E = bsdw.energy_function
bsdw_F = bsdw.force_function

bsdw_state_A = circular_state(center=(-2.,0), axes=(1,1), angle=0, radius=0.15)
bsdw_state_B = circular_state(center=(2.,0), axes=(1,1), angle=0, radius=0.15)

## Input and output folders

`weights` is the trajectory weight used to reweight AAA-TPS: `acc_w` (the weight of the accepted
shooting point) for uniform selection and `1/acc_p_sel_norm` (inverse normalization of the selection
probability) for Gaussian selection. Methods with `weights=None` sample the transition path ensemble directly.

In [ ]:
inputdir_ref = "./output/twoway_shooting/bsdw/Uniform"
outputdir_ref = "./reference/twoway_shooting/bsdw/Uniform"

methods = {
    "2w_Gaus": dict(inputdir="./output/twoway_shooting/bsdw/Gaussian", outputdir="./results/twoway_shooting/bsdw/Gaussian",
                    run="2w", label="Two-Way (Gaussian)", weights=None),
    "1w_Gaus": dict(inputdir="./output/oneway_shooting/bsdw/Gaussian", outputdir="./results/oneway_shooting/bsdw/Gaussian",
                    run="1w", label="One-Way (Gaussian)", weights=None),
    "ARA_TPS_Unif": dict(inputdir="./output/ARA-TPS/bsdw/Uniform", outputdir="./results/ARA_TPS/bsdw/Uniform",
                         run="ARA-TPS", label="ARA-TPS (Uniform)", weights=None),
    "ARA_TPS_Gaus": dict(inputdir="./output/ARA-TPS/bsdw/Gaussian", outputdir="./results/ARA_TPS/bsdw/Gaussian",
                         run="ARA-TPS", label="ARA-TPS (Gaussian)", weights=None),
    "AAA_TPS_Unif": dict(inputdir="./output/AAA-TPS/bsdw/Uniform", outputdir="./results/AAA_TPS/bsdw/Uniform",
                         run="AAA-TPS", label="AAA-TPS (Uniform)", weights="acc_w"),
    "AAA_TPS_Gaus": dict(inputdir="./output/AAA-TPS/bsdw/Gaussian", outputdir="./results/AAA_TPS/bsdw/Gaussian",
                         run="AAA-TPS", label="AAA-TPS (Gaussian)", weights="inv_acc_p_sel_norm"),
}

n_replicas_ref = 250
n_replicas = 50

for m in methods.values():
    os.makedirs(m["outputdir"], exist_ok=True)
os.makedirs(outputdir_ref, exist_ok=True)
os.makedirs("./results/path_lengths", exist_ok=True)

In [ ]:
def replica_dir(inputdir, replica):
    return os.path.join(inputdir, f"R_{replica:04d}")


def load_weights(m, replica, kind):
    """Trajectory weights of AAA-TPS for the trajectory ensemble (kind='te') or the path lengths (kind='len')."""
    rdir = replica_dir(m["inputdir"], replica)
    if m["weights"] == "acc_w":
        return np.loadtxt(os.path.join(rdir, f"acc_w_{kind}_{m['run']}.txt"))
    if m["weights"] == "inv_acc_p_sel_norm":
        return 1/np.loadtxt(os.path.join(rdir, f"acc_p_sel_norm_{kind}_{m['run']}.txt"))
    raise ValueError(f"No weights for {m['label']}")


def load_force_evaluations(m, replica):
    return np.loadtxt(os.path.join(replica_dir(m["inputdir"], replica), f"force_evaluations_{m['run']}.txt"))

## Reference $p(x|\mathrm{TP})$

Two-way shooting with uniform selection probability, 250 replicas. The histogram covers the whole
plotting region of Fig. 2B.

In [ ]:
x_min, x_max = -3.3, 3.3
y_min, y_max = -3.3, 3.3
num_bins = 100

Hist_ref = []

for replica in tqdm(range(n_replicas_ref)):

    with open(os.path.join(replica_dir(inputdir_ref, replica), 'paths_2w.pkl'), 'rb') as f:
        trajectory_ensemble_ref = pickle.load(f)

    all_positions_ref = np.vstack(trajectory_ensemble_ref)

    H_ref, xedges, yedges = np.histogram2d(
        all_positions_ref[:, 0],
        all_positions_ref[:, 1],
        bins=num_bins,
        range=[[x_min, x_max], [y_min, y_max]],
        density=True
    )

    Hist_ref.append(H_ref)
    del trajectory_ensemble_ref, all_positions_ref  # free memory before loading the next replica

Hist_ref_avg = np.array(Hist_ref).mean(axis=0)
np.savetxt(os.path.join(outputdir_ref, 'Hist_ref_avg_for_plotting.txt'), Hist_ref_avg)

In [ ]:
fig, ax = plt.subplots(1, figsize=(5,4), dpi=180)

plt.imshow(Hist_ref_avg.T, origin='lower', extent=[x_min, x_max, y_min, y_max], cmap='hot')
plt.colorbar(label=r'$P(x|\mathrm{TP})$')
plt.xlabel(r'$x^0$')
plt.ylabel(r'$x^1$')
plt.title(r'Reference (Two-Way Uniform $p_{\text{sel}}$)')
plt.show()

## Acceptance rate and effective sample size (Table S1)

The acceptance rate of each replica is read from `acceptance_<run>.txt`; the relative effective
sample size $\mathrm{ESS}/N$ of AAA-TPS is computed from the normalized trajectory weights.
Values are averaged over replicas (error: standard error of the mean).

In [ ]:
efficiency = {}

for key, m in methods.items():

    if m["weights"] is None:
        acc = np.zeros(n_replicas)
        for replica in range(n_replicas):
            with open(os.path.join(replica_dir(m["inputdir"], replica), f"acceptance_{m['run']}.txt"), 'r') as f:
                acc[replica] = float(f.readline().split()[-1])
        efficiency[key] = dict(r_acc=acc.mean(), r_acc_err=acc.std(ddof=1)/np.sqrt(n_replicas))
        print(f"{m['label']}: r_acc = {acc.mean():.4f} +- {acc.std(ddof=1)/np.sqrt(n_replicas):.4f}")
    else:
        ress = np.zeros(n_replicas)
        for replica in range(n_replicas):
            weights = load_weights(m, replica, "te")
            ress[replica] = relative_effective_sample_size(weights/weights.sum())
        efficiency[key] = dict(ess=ress.mean(), ess_err=ress.std(ddof=1)/np.sqrt(n_replicas))
        print(f"{m['label']}: ESS/N = {ress.mean():.4f} +- {ress.std(ddof=1)/np.sqrt(n_replicas):.4f}")

## Autocorrelation of the transition channel (Fig. 2D)

Each sampled trajectory is assigned to the upper ($+1$) or lower ($-1$) channel according to the sign
of its average $x^1$. The normalized autocorrelation function of this indicator is computed for each
replica and averaged over replicas. For AAA-TPS the autocorrelation is computed both without and with
the trajectory weights.

The autocorrelation is computed up to `max_lag` trials, far beyond its decay
(the figure shows up to $10^6$ force evaluations, i.e. less than $2\cdot10^3$ trials).

Output (`ch_correlation_<run>.txt`): estimated force evaluations (lag times the average force evaluations
per trial), autocorrelation and, for AAA-TPS, reweighted autocorrelation.

In [ ]:
max_lag = 10000

ch_correlation = {}

for key, m in methods.items():

    acf_ch = np.zeros((n_replicas, max_lag))
    acf_ch_rw = np.zeros((n_replicas, max_lag))
    force_eval = []
    ratio_up = np.zeros(n_replicas)

    for replica in range(n_replicas):

        with open(os.path.join(replica_dir(m["inputdir"], replica), f"paths_{m['run']}.pkl"), 'rb') as f:
            trajectory_ensemble = pickle.load(f)

        force_eval.append(load_force_evaluations(m, replica))

        # Channel of each path: sign of the average x^1
        path_sgn_y = np.sign(np.array([path.mean(axis=0)[1] for path in trajectory_ensemble]))
        ratio_up[replica] = (path_sgn_y > 0).sum() / len(path_sgn_y)
        del trajectory_ensemble

        acf_ch[replica] = get_weighted_autocorrelation_function(path_sgn_y, norm=True, mean=True, max_lag=max_lag)

        if m["weights"] is not None:
            weights = load_weights(m, replica, "te")
            acf_ch_rw[replica] = get_weighted_autocorrelation_function(path_sgn_y, weights=weights/weights.sum(),
                                                                       norm=True, mean=True, max_lag=max_lag)

    avg_per_tr_force_eval = np.concatenate(force_eval).mean()
    est_f_eval = np.arange(max_lag)*avg_per_tr_force_eval

    columns = [est_f_eval, acf_ch.mean(axis=0)]
    if m["weights"] is not None:
        columns.append(acf_ch_rw.mean(axis=0))
    ch_correlation[key] = np.c_[tuple(columns)]

    np.savetxt(os.path.join(m["outputdir"], f"ch_correlation_{m['run']}.txt"), ch_correlation[key])

    print(f"{m['label']}: fraction of paths in the upper channel = {ratio_up.mean():.3f} +- {ratio_up.std(ddof=1)/np.sqrt(n_replicas):.3f}, "
          f"force evaluations per trial = {avg_per_tr_force_eval:.1f}")

Integrated autocorrelation (Table S1 and main text): $\tau^{\mathrm{ch}} = 2\int C_{\mathrm{ch}}(\tau)\,d\tau$
truncated at $10^3$ trials, and the corresponding number of force evaluations
$\tilde{n}_{\mathrm{F}}^{\mathrm{ch}}$ needed for an independent sample. For AAA-TPS the reweighted
autocorrelation (last column) is used, i.e. the correlation in the target transition path ensemble.

In [ ]:
icut = 1000

for key, m in methods.items():
    data = ch_correlation[key]
    tau_ch = 2*np.trapezoid(data[:icut+1, -1], dx=1)
    efficiency[key].update(tau_ch=tau_ch, nF_ch=tau_ch*data[1, 0])
    print(f"{m['label']}: tau_ch = {tau_ch:.1f}, n_F^ch = {tau_ch*data[1, 0]:.3e}")

In [ ]:
fig, ax = plt.subplots(1, figsize=(10 * 0.39, 7.5 * 0.39), dpi=180)

for key, m in methods.items():
    data = ch_correlation[key]
    ax.plot(data[:, 0], data[:, -1], label=m["label"])

ax.axhline(y=0, color="k", ls="--", linewidth=1, alpha=0.5)
ax.set_xscale("log")
ax.set_xlim(1e3, 1e6)
ax.set_ylim(-0.1, 1)
ax.set_xlabel(r"Force Evaluations $n_\text{F}$")
ax.set_ylabel(r"$C_{\text{ch}}(n_{\text{F}})$")
ax.legend(frameon=False, fontsize=6)
plt.show()

## Efficiency summary (Table S1)

In [ ]:
with open("./results/efficiency_bsdw.txt", "w") as f:
    f.write("# Bi-stable double well: acceptance rate r_acc, relative effective sample size ESS/N (mean and standard error over replicas),\n")
    f.write("# integrated channel autocorrelation tau_ch (trials, reweighted for AAA-TPS) and force evaluations per independent sample nF_ch\n")
    f.write(f"# {'method':<14s} {'r_acc':>8s} {'r_acc_err':>9s} {'ESS/N':>8s} {'ESS/N_err':>9s} {'tau_ch':>8s} {'nF_ch':>11s}\n")
    for key, e in efficiency.items():
        f.write(f"{key:<16s} {e.get('r_acc', np.nan):8.4f} {e.get('r_acc_err', np.nan):9.4f} {e.get('ess', np.nan):8.4f} "
                f"{e.get('ess_err', np.nan):9.4f} {e['tau_ch']:8.1f} {e['nF_ch']:11.4e}\n")

print(open("./results/efficiency_bsdw.txt").read())

## Path length distribution (Fig. S6, right)

Histograms of the path lengths $L$ of each replica, averaged over replicas, on bins
$L_b = (b/199)^{10} L_{\mathrm{max}}$. For AAA-TPS both the raw (unreweighted) and the reweighted
distributions are computed.

In [ ]:
lengths_ref = [np.loadtxt(os.path.join(replica_dir(inputdir_ref, replica), "lengths_2w.txt")) for replica in range(n_replicas_ref)]

lengths = {}
weights_lengths = {}
for key, m in methods.items():
    lengths[key] = [np.loadtxt(os.path.join(replica_dir(m["inputdir"], replica), f"lengths_{m['run']}.txt")) for replica in range(n_replicas)]
    if m["weights"] is not None:
        weights_lengths[key] = []
        for replica in range(n_replicas):
            w = load_weights(m, replica, "len")
            weights_lengths[key].append(w/w.sum())

max_length = max(max(l.max() for l in lengths_ref), max(l.max() for key in lengths for l in lengths[key]))

n_bins = 200
power = 10
bins = np.linspace(0, 1, n_bins) ** power * max_length
bin_centers = (bins[:-1] + bins[1:])/2

In [ ]:
def average_histogram(samples, weights=None):
    hists = [np.histogram(s, bins=bins, density=True, weights=None if weights is None else weights[i])[0] for i, s in enumerate(samples)]
    return np.array(hists).mean(axis=0)

P_L = {"Reference": average_histogram(lengths_ref)}
for key, m in methods.items():
    if m["weights"] is None:
        P_L[key] = average_histogram(lengths[key])
    else:
        P_L[f"{key}_unrew"] = average_histogram(lengths[key])
        P_L[key] = average_histogram(lengths[key], weights_lengths[key])

np.savetxt("./results/path_lengths/bsdw_P_L.txt", np.c_[(bins[:-1], bins[1:]) + tuple(P_L.values())],
           header="bin_left bin_right " + " ".join(P_L.keys()))

In [ ]:
fig, ax = plt.subplots(1, figsize=(10 * 0.39, 7.5 * 0.39), dpi=180)

for name, hist in P_L.items():
    ax.plot(bin_centers, hist, ls=":" if name.endswith("unrew") else "-", label=name)

ax.set_xlabel(r"$L$")
ax.set_ylabel(r"$P(L)$")
ax.set_xlim(0, max_length)
plt.title("Bi-Stable Double Well")
plt.legend(frameon=False, fontsize=6)
plt.show()